# 02 - Bronze Layer: Autoloader Ingestion

**Project:** Retail Analytics & Product Dimension History

## What this notebook does
Ingests all four star-schema source files (Customers, Products, Stores,
Transactions) into bronze Delta tables using Autoloader, same
trigger(availableNow=True) batch-style pattern used in every prior project.

## Tables created
- `main.retail_analytics.bronze_customers`
- `main.retail_analytics.bronze_products`
- `main.retail_analytics.bronze_stores`
- `main.retail_analytics.bronze_transactions`

In [0]:
from pyspark.sql.types import StructType
from pyspark.sql.functions import current_timestamp, col

def autoload_csv_to_bronze(source_path: str, table_name: str, schema: StructType):
    checkpoint_path = f"/Volumes/main/retail_analytics/checkpoints/{table_name}"
    schema_path = f"/Volumes/main/retail_analytics/checkpoints/{table_name}_schema"
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", "true")
        .schema(schema)
        .load(source_path)
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )
    (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(f"main.retail_analytics.{table_name}")
    )

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, DoubleType, IntegerType

customers_schema = StructType([
    StructField("CustomerID", StringType(), True),
    StructField("FirstName", StringType(), True),
    StructField("LastName", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("BirthDate", DateType(), True),
    StructField("City", StringType(), True),
    StructField("JoinDate", DateType(), True),
])

products_schema = StructType([
    StructField("ProductID", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("SubCategory", StringType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("CostPrice", DoubleType(), True),
])

stores_schema = StructType([
    StructField("StoreID", StringType(), True),
    StructField("StoreName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Region", StringType(), True),
])

transactions_schema = StructType([
    StructField("TransactionID", StringType(), True),
    StructField("Date", DateType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("StoreID", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("PaymentMethod", StringType(), True),
])

In [0]:
autoload_csv_to_bronze(
    source_path="/Volumes/main/retail_analytics/raw_data/Customers*.csv",
    table_name="bronze_customers",
    schema=customers_schema,
)

autoload_csv_to_bronze(
    source_path="/Volumes/main/retail_analytics/raw_data/Products*.csv",
    table_name="bronze_products",
    schema=products_schema,
)

autoload_csv_to_bronze(
    source_path="/Volumes/main/retail_analytics/raw_data/Stores*.csv",
    table_name="bronze_stores",
    schema=stores_schema,
)

autoload_csv_to_bronze(
    source_path="/Volumes/main/retail_analytics/raw_data/Transactions*.csv",
    table_name="bronze_transactions",
    schema=transactions_schema,
)

In [0]:
%sql
SELECT COUNT(*) FROM main.retail_analytics.bronze_products;

COUNT(*)
50


In [0]:
%sql
SELECT _source_file, COUNT(*) 
FROM main.retail_analytics.bronze_products 
GROUP BY _source_file;

_source_file,COUNT(*)
/Volumes/main/retail_analytics/raw_data/Products.csv,50


In [0]:
%sql
SELECT 'bronze_customers' AS table_name, COUNT(*) AS row_count FROM main.retail_analytics.bronze_customers
UNION ALL
SELECT 'bronze_products', COUNT(*) FROM main.retail_analytics.bronze_products
UNION ALL
SELECT 'bronze_stores', COUNT(*) FROM main.retail_analytics.bronze_stores
UNION ALL
SELECT 'bronze_transactions', COUNT(*) FROM main.retail_analytics.bronze_transactions;

table_name,row_count
bronze_customers,200
bronze_products,50
bronze_stores,5
bronze_transactions,5000
